In [0]:
%sql
create table if not exists project_etl.delta_lake.stock_prices

In [0]:
%sql
use catalog project_etl;
use schema landing;
select current_catalog();
create external volume if not exists opertaional
location 's3://data-databricks-sample-102426687040/project_con/landing/stockprice/';

In [0]:
%sql
-- %python
-- -- %python
-- -- ---*********Only SQL statements is giving a issue
-- -- -- SET spark.sql.files.ignoreCorruptFiles = true;
-- -- -- copy into project_etl.delta_lake.stock_prices
-- -- -- from '/Volumes/project_etl/landing/stock_prices'
-- -- -- fileformat = json
-- -- -- format_options ('inferSchema' = 'true')
-- -- -- copy_options ('mergeSchema'= 'true');



In [0]:
%sql
SELECT *
FROM read_files(
  'dbfs:/Volumes/project_etl/landing/stock_prices/*json',
  format => 'json',
  multiline => 'true'
);



In [0]:
%sql
COPY INTO project_etl.delta_lake.stock_prices
FROM '/Volumes/project_etl/landing/stock_prices'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
# Read JSON files with schema inference and ignore corrupt files
stock_df = (
    spark.read.format("json")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("ignoreCorruptFiles", "true")
    .load("/Volumes/project_etl/landing/stock_prices")
)

# Write to Delta table with schema merge
stock_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("project_etl.delta_lake.stock_prices")